# Watchability Score Engine — Data Pull & Label Construction

This notebook builds the **target variable (`y`)**: a 0–100 `watchability_score` for every *completed* NFL game, 2000–2026.

**Why we need to build `y` at all:** this is supervised regression (Phase 2 — Linear Regression / Core ML). A model can only learn `f(features) → y` if `y` exists. Nobody publishes a "how watchable was this game" number, so we construct one from hindsight (what actually happened in the game), then later train a model to *predict* it from pre-game information only.

**The one rule that governs this whole notebook:** everything computed here uses *outcome* data (final scores, win-probability swings). That makes these columns **label ingredients, never model features.** Using them as features would be target leakage — the same failure mode as fitting a scaler on test data (Phase 2 — ML Pipelines), just more blatant.

Pipeline: `schedules` + `play-by-play` → `src/drama_features.py` → `master_games.csv` → `src/build_label.py` → `labeled_games.csv`

In [1]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import nfl_data_py as nfl

from drama_features import compute_drama
from build_label import build_label, completed_games, zscore, DEFAULT_WEIGHTS

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

YEARS = list(range(2000, 2027))
DATA = '../data'

## 1. Game-level schedule data

One row per game: teams, scores, `game_type` (REG/WC/DIV/CON/SB), `div_game`, betting lines, weather, QBs, coaches.

Cached to CSV — the pull only happens if the file is missing.

In [2]:
sched_path = f'{DATA}/games_2000_2026.csv'
if os.path.exists(sched_path):
    games = pd.read_csv(sched_path, low_memory=False)
else:
    games = nfl.import_schedules(YEARS)
    games.to_csv(sched_path, index=False)

print('Shape:', games.shape)
print(games['game_type'].value_counts().to_string())

Shape: (7289, 46)
game_type
REG    6991
WC      116
DIV     104
CON      52
SB       26


## 2. Play-by-play win probability

Each play has nflverse's model-estimated win probability and **WPA** (win probability added = change in WP caused by that play).

All needed columns are pulled **in one request** (an earlier version re-downloaded 1.2M rows three times as new columns turned out to be needed).

### Column-semantics traps (each one caused a real bug here)
| column | what you'd assume | what it actually is |
|---|---|---|
| `wp` | home team's WP | WP of **whoever has the ball** — flips every possession |
| `home_score` / `away_score` | running score | the **final** score, repeated on every row |
| `total_home_score` / `total_away_score` | — | the actual running score |

So anything measuring leads or comebacks uses `home_wp` (a fixed reference frame), not `wp`.

In [3]:
pbp_path = f'{DATA}/pbp_wp_2000_2026.parquet'
PBP_COLS = ['game_id', 'season', 'wpa', 'wp', 'home_wp', 'away_wp', 'qtr', 'play_type',
            'game_seconds_remaining', 'home_score', 'away_score',
            'total_home_score', 'total_away_score']
if os.path.exists(pbp_path):
    pbp = pd.read_parquet(pbp_path)
else:
    pbp = nfl.import_pbp_data(YEARS, columns=PBP_COLS, include_participation=False, downcast=True)
    pbp.to_parquet(pbp_path, index=False)

print('Shape:', pbp.shape, '| games covered:', pbp['game_id'].nunique())

Shape: (1238981, 13) | games covered: 7047


## 3. Drama metrics (`src/drama_features.py`)

### 3a. Why raw Σ|WPA| was not enough
The first excitement metric was simply the sum of |WPA| over all plays. It measures **churn, not drama**: a stalled, punt-heavy game produces hundreds of tiny WP wobbles that add up. Verified on known games:

- **SB LIII** (13–3 NE over LA, famously dull) → Σ|WPA| ≈ **4.95**
- **SB LI** (NE comes back from 28–3) → Σ|WPA| ≈ **3.06**

The snoozer out-scored the greatest comeback in Super Bowl history. Summary stats (`.describe()`) looked perfectly reasonable — only checking against **ground-truth games** exposed it.

### 3b. Two fixes
1. **`excitement_weighted`** — each play's |WPA| is weighted by `1 + (fraction of game elapsed)`: 1.0× at kickoff → 2.0× at the final whistle. A 4th-quarter swing matters more than 1st-quarter noise. (Plain-language: this is just a **dot product** — Phase 1 linear algebra — between the vector of |WPA| values and a vector of time weights.)
2. **`comeback_factor`** — find the lowest WP the *eventual winner* ever had, then score `log(1 + 1/min_wp)`. Being at 0.5% and winning is rare and memorable.
   - **Why the log:** `1/0.005 = 200` vs `1/0.5 = 2` — a 100× gap would let a single comeback dominate. The log compresses that to ~5.3 vs ~1.1, the same instinct as log-scaling a skewed feature before a linear model.

In [4]:
drama_path = f'{DATA}/game_drama_features.csv'
if os.path.exists(drama_path):
    drama = pd.read_csv(drama_path)
else:
    drama = compute_drama(pbp)
    drama.to_csv(drama_path, index=False)

print('Shape:', drama.shape)
drama.drop(columns='game_id').describe().round(3)

Shape: (7047, 6)


,excitement_index,excitement_weighted,max_swing,winner_min_wp,comeback_factor
count,7047.000,7047.000,7047.000,7047.000,7047.000
mean,3.659,5.536,0.227,0.330,1.635
std,1.486,2.630,0.133,0.152,0.932
min,0.597,0.680,0.047,0.000,0.939
25%,2.495,3.436,0.136,0.218,1.168
50%,3.610,5.344,0.188,0.350,1.351
75%,4.685,7.302,0.276,0.451,1.721
max,11.278,19.851,1.000,0.643,6.909


## 4. Master table

Merge the drama metrics onto the schedule and add the three schedule-based signals:

| signal | source | raw scale |
|---|---|---|
| `playoff_stakes` | `game_type` → REG 0, WC 1, DIV 2, CON 3, SB 4 | 0–4 ordinal |
| `rivalry` | `div_game` | 0/1 |
| `fireworks` | `total` (combined points) | ~0–100 |
| `excitement_weighted` | play-by-play | ~0.5–20 |
| `comeback_factor` | play-by-play | ~0.7–6.9 |

`playoff_stakes` is an ordinal where **order genuinely means something** (SB > CON > DIV), so mapping to integers is fine here — unlike `division`, where label encoding is wrong (Phase 2 — Feature Encoding, the R²=0.18 vs 0.97 demo).

In [5]:
master = games.merge(drama, on='game_id', how='left')
master['playoff_stakes'] = master['game_type'].map({'REG': 0, 'WC': 1, 'DIV': 2, 'CON': 3, 'SB': 4})
master['rivalry'] = master['div_game']
master['fireworks'] = master['total']
master.to_csv(f'{DATA}/master_games.csv', index=False)

played = master['home_score'].notna()
print('Master:', master.shape)
print('Played games:', played.sum(), '| unplayed (future):', (~played).sum())
print('Played but missing pbp drama:', (played & master['excitement_weighted'].isna()).sum())

Master: (7289, 54)
Played games: 7049 | unplayed (future): 240
Played but missing pbp drama: 2


## 5. Scaling: why percentile rank beat z-score

The five signals are on wildly different scales (0/1 vs 0–4 vs 0–100). Combining them raw would let `fireworks` dominate purely because its numbers are bigger, just as unscaled features wreck gradient descent (Phase 1 — Optimization).

**First attempt: z-score** (`(x − mean) / std`, the standard move from Phase 2). It failed in a subtle way:
- ~96% of games are regular season (`playoff_stakes = 0`), so the std of `playoff_stakes` is tiny.
- A Super Bowl's z-score is therefore enormous. The z-score turns **rarity into magnitude**.

Live comparison below: the highest-ever z-score on `excitement_weighted` vs a Super Bowl's `playoff_stakes` z-score.

In [6]:
done = completed_games(master)
z_sb = zscore(done['playoff_stakes']).max()
z_ex = zscore(done['excitement_weighted']).max()
print(f'z-score of a Super Bowl on playoff_stakes:          {z_sb:.2f}')
print(f'z-score of the most exciting game in 27 seasons:    {z_ex:.2f}')
print(f'share of games that are regular season:            {(done.playoff_stakes == 0).mean():.1%}')

z-score of a Super Bowl on playoff_stakes:          8.89
z-score of the most exciting game in 27 seasons:    5.44
share of games that are regular season:            95.8%


Under z-scoring, *any* Super Bowl — including the 13–3 snoozer — outranks every regular-season game ever played, because the stakes term alone swamps everything else.

**Fix: percentile rank** for the continuous signals (each game's position 0→1 among all games). Percentiles are bounded, so one extreme game can't stretch the scale, and they're robust to skew. `playoff_stakes` is divided by 4 and `rivalry` is already 0/1, so everything lives on [0, 1].

Weights (a **taste judgment**, still open to revision):

In [7]:
pd.Series(DEFAULT_WEIGHTS, name='weight').to_frame()

,weight
excitement_weighted,0.335
comeback_factor,0.170
fireworks,0.170
playoff_stakes,0.275
rivalry,0.050


## 6. Build the label

`watchability_score = 100 × Σ(weight × scaled_signal) / Σweights`. That's a weighted sum, literally a dot product of the weight vector with each game's scaled-signal vector, the same form as a linear model's prediction. The difference is that here **we** chose the weights by hand. The model's job later is to predict this number from pre-game data it can actually see.

In [8]:
labeled = build_label(master)
labeled['rank'] = labeled['watchability_score'].rank(ascending=False, method='min').astype(int)
labeled.to_csv(f'{DATA}/labeled_games.csv', index=False)

print('Labeled games:', len(labeled))
labeled['watchability_score'].describe().round(2)

Labeled games: 7047


count    7047.00
mean       36.18
std        15.01
min         2.57
25%        23.92
50%        35.71
75%        47.76
max        87.46
Name: watchability_score, dtype: float64

## 7. Sanity check against ground truth

Summary stats hid every bug so far, so we check the score against specific games whose watchability we already know.

In [9]:
known = {
    '2016_21_NE_ATL': 'SB LI — 28-3 comeback',
    '2018_21_NE_LA':  'SB LIII — 13-3 snoozer',
    '2021_20_BUF_KC': 'BUF-KC 13 seconds (DIV)',
    '2018_11_KC_LA':  'KC-LA 54-51 MNF',
}
cols = ['game_id', 'home_score', 'away_score', 'excitement_index', 'excitement_weighted', 'comeback_factor',
        'watchability_score', 'rank']
chk = labeled[labeled['game_id'].isin(known)][cols].copy()
chk.insert(1, 'game', chk['game_id'].map(known))
print(f'out of {len(labeled)} games')
chk.drop(columns='game_id').round(2)

out of 7047 games


,game,home_score,away_score,excitement_index,excitement_weighted,comeback_factor,watchability_score,rank
4522,SB LI — 28-3 comeback,28.0,34.0,3.06,4.75,5.20,73.52,22
4950,KC-LA 54-51 MNF,54.0,51.0,6.86,11.39,1.80,63.07,231
5056,SB LIII — 13-3 snoozer,3.0,13.0,4.95,7.59,1.41,63.21,218
5874,BUF-KC 13 seconds (DIV),42.0,36.0,7.29,12.45,2.11,78.25,10


**Read this table honestly** (weights as of 2026-09-24, rivalry = 0.05):
- **Time-weighting did *not* fix the churn problem.** SB LIII still has the *higher* `excitement_weighted` (7.59 vs 4.75). SB LI ranks far above it (#22 vs #218) only because of `comeback_factor` (5.20 vs 1.41).
- **Cutting rivalry from 0.15 → 0.05 moved the famous games toward where intuition puts them:** SB LI #190 → #22, BUF–KC "13 seconds" #64 → #10. Under the old weight, ~37% of all games (division games) got a flat bonus that pushed everything else down.
- **Remaining quirk:** the 13–3 SB LIII (#218) still edges out the 54–51 KC–LA shootout (#231), because playoff stakes carry 27.5%. That's a taste call, parked rather than refined endlessly.

In [10]:
print('Top 10 most watchable games, 2000-2026:')
labeled.nsmallest(10, 'rank')[['rank', 'season', 'game_type', 'away_team', 'away_score',
                               'home_team', 'home_score', 'watchability_score']]

Top 10 most watchable games, 2000-2026:


,rank,season,game_type,away_team,away_score,home_team,home_score,watchability_score
6446,1,2023,SB,SF,22.0,KC,25.0,87.46
1851,2,2006,CON,NE,34.0,IND,38.0,83.84
3988,3,2014,SB,NE,28.0,SEA,24.0,83.36
6161,4,2022,SB,KC,38.0,PHI,35.0,82.50
5055,5,2018,CON,NE,37.0,KC,31.0,80.98
4789,6,2017,SB,PHI,41.0,NE,33.0,80.52
3448,7,2012,DIV,BAL,38.0,DEN,35.0,80.10
2652,8,2009,CON,MIN,28.0,NO,31.0,78.93
5323,9,2019,SB,SF,20.0,KC,31.0,78.48
5874,10,2021,DIV,BUF,36.0,KC,42.0,78.25


## 8. How the signals relate to each other

Pearson correlation between the five ingredients (Phase 1 — Statistics). Low correlation means each signal adds *new* information to the composite rather than double-counting.

The CLAUDE.md brief guessed `rivalry` and `playoff_stakes` would be positively correlated. The data says the opposite: playoff games are rarely division matchups.

In [11]:
sig = list(DEFAULT_WEIGHTS)
done[sig].corr().round(3)

,excitement_weighted,comeback_factor,fireworks,playoff_stakes,rivalry
excitement_weighted,1.000,0.357,0.033,0.007,0.007
comeback_factor,0.357,1.000,0.073,0.005,0.025
fireworks,0.033,0.073,1.000,0.037,-0.054
playoff_stakes,0.007,0.005,0.037,1.000,-0.102
rivalry,0.007,0.025,-0.054,-0.102,1.000


## 9. Where this leaves us / next steps

**Done:** `labeled_games.csv` has `y` for every completed game.

**Next notebook (`02_features_and_baseline`):**
1. **Pre-game features only.** Things knowable before kickoff: rolling team form (last-N point differential, win %), both teams' records, `spread_line` / `total_line` (the Vegas total is basically the market's forecast of `fireworks`), `div_game`, `game_type`, week, rest days, QB, weather/roof. None of the Section 3 drama columns.
2. **Walk-forward validation** (Phase 2 — Time Series): train on seasons < S, test on season S. No random splits.
3. **Baselines first:** predict the mean, then Ridge in a `Pipeline`, then LightGBM/XGBoost with early stopping. You can't tell whether a boosted model is good without a floor to compare it against.
4. Because this is a *ranking* problem at heart ("which game this week?"), also evaluate **within-week rank correlation (Spearman)** alongside RMSE/R².